# Hito 1 — F1 Race Strategy Advisor: Baseline

**Team:** Group 18 — Alonso Cárdenas, Benjamín Sánchez  
**Target:** `is_top10` (locked)  
**Split:** Train 2019–2021 / Calibration 2022 / Test 2023–2024  
**Primary metric:** Brier Score

In [ ]:
# Cell 1 - Imports & seed
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, f1_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('Imports OK - sklearn with FrozenEstimator for calibration')

In [ ]:
# Cell 2 - Load data
df = pd.read_csv('data/f1_strategy_race_level.csv')
print(f'Shape: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'\nis_top10 distribution:\n{df["is_top10"].value_counts(normalize=True).round(3)}')
df.head(3)

In [ ]:
# Cell 3 - Temporal split (LOCKED)
train = df[df['season'].isin([2019, 2020, 2021])].copy()
cal   = df[df['season'] == 2022].copy()
test  = df[df['season'].isin([2023, 2024])].copy()

print(f'Train: {len(train)} rows, seasons {sorted(train["season"].unique())}')
print(f'Cal:   {len(cal)} rows, seasons {sorted(cal["season"].unique())}')
print(f'Test:  {len(test)} rows, seasons {sorted(test["season"].unique())}')
print(f'\nTotal: {len(train)+len(cal)+len(test)} (of {len(df)})')
print(f'\nis_top10 rates - Train: {train["is_top10"].mean():.3f}, Cal: {cal["is_top10"].mean():.3f}, Test: {test["is_top10"].mean():.3f}')

## Leakage Audit

Every column in the dataset classified into one of three categories.

In [ ]:
# Cell 4 - Leakage audit
leakage_audit = {
    'pre_race': [
        'season', 'round', 'race_name', 'circuit_id', 'circuit', 'circuit_type',
        'driver_id', 'driver_name', 'Driver', 'Team', 'constructor_name',
        'grid_position', 'qualifying_position',
        'driver_prior3_avg_finish', 'constructor_prior3_avg_finish',
        'driver_circuit_prior_avg', 'constructor_tier'
    ],
    'scenario_inputs': [
        'n_stops', 'strategy_type', 'compound_sequence',
        'stint_lengths', 'stint1_length', 'stint2_length', 'stint3_length',
        'stint4_length', 'stint5_length', 'avg_pit_stop_duration_s',
        'total_pit_time_s', 'first_pit_lap', 'last_pit_lap'
    ],
    'post_race_audit_only': [
        'qualifying_time_s',
        'track_status_summary', 'safety_car_periods', 'safety_car_laps',
        'vsc_laps', 'weather_actual', 'wet_laps',
        'avg_track_temp', 'avg_air_temp',
        'finish_position', 'points', 'positions_gained',
        'is_top3', 'is_top5', 'is_top10', 'dnf', 'status'
    ]
}

print('=== LEAKAGE AUDIT ===')
for cat, cols in leakage_audit.items():
    print(f'\n{cat.upper()} ({len(cols)} columns):')
    for c in cols:
        exists = '[OK]' if c in df.columns else '[MISSING]'
        print(f'  {exists} {c}')

all_audited = set()
for cols in leakage_audit.values():
    all_audited.update(cols)
missing = set(df.columns) - all_audited
print(f'\nUnaudited columns: {missing if missing else "None - all columns classified"}')

print('\n--- KEY DECLARATION ---')
print('Strategy features (n_stops, compound_sequence, stint_lengths, etc.) are POST-RACE')
print('observations in the raw data. For this capstone they are ALLOWED as SCENARIO INPUTS')
print('because the product is a what-if comparison tool. The user intentionally sets these')
print('variables. They are NOT pre-race predictions.')
print('\nSafety car, weather, and track status columns are AUDIT-ONLY. They cannot be used')
print('as predictors - only for post-hoc error analysis or scenario stress tests.')

In [ ]:
# Cell 5 - Feature engineering
FEATURES = ['grid_position', 'constructor_tier', 'n_stops']
TARGET = 'is_top10'

def prepare_features(data):
    """Prepare features for modeling."""
    out = data.copy()
    # Clip grid_position: pit-lane starts (0) mapped to 20
    out['grid_position'] = out['grid_position'].clip(lower=1, upper=20).fillna(20)
    # Encode constructor_tier as ordinal (front > midfield > backmarker)
    tier_map = {'front': 2, 'midfield': 1, 'backmarker': 0}
    out['constructor_tier'] = out['constructor_tier'].map(tier_map).fillna(1)
    # n_stops: fill missing with median
    out['n_stops'] = out['n_stops'].fillna(out['n_stops'].median())
    return out

train_p = prepare_features(train)
cal_p   = prepare_features(cal)
test_p  = prepare_features(test)

X_train, y_train = train_p[FEATURES], train_p[TARGET]
X_cal,   y_cal   = cal_p[FEATURES],   cal_p[TARGET]
X_test,  y_test  = test_p[FEATURES],  test_p[TARGET]

print(f'Feature shapes - Train: {X_train.shape}, Cal: {X_cal.shape}, Test: {X_test.shape}')
print(f'\nFeature summary (train):')
X_train.describe().round(2)

## Baseline 1: Grid-Rule Heuristic

In [ ]:
# Cell 6 - Grid-rule heuristic baseline
def grid_rule_baseline(grid_positions):
    """P(top10) = 0.85 if grid <= 10, 0.20 otherwise.
    F1-defensible: top-10 qualifiers score points ~85% of the time historically."""
    return np.where(grid_positions <= 10, 0.85, 0.20)

y_prob_grid = grid_rule_baseline(X_test['grid_position'].values)

brier_grid = brier_score_loss(y_test, y_prob_grid)
logloss_grid = log_loss(y_test, y_prob_grid)
auc_grid = roc_auc_score(y_test, y_prob_grid)

print('=== Grid-Rule Heuristic (Test Set 2023-2024) ===')
print(f'Brier Score: {brier_grid:.4f}')
print(f'Log Loss:    {logloss_grid:.4f}')
print(f'ROC-AUC:     {auc_grid:.4f}')
print(f'\nDocent grid-rule floor: Brier = 0.208')
if brier_grid < 0.208:
    print(f'Our grid-rule: Brier = {brier_grid:.4f} -> BEATS docent grid-rule')
else:
    print(f'Our grid-rule: Brier = {brier_grid:.4f} -> below docent grid-rule')

## Baseline 2: Calibrated Logistic Regression

In [ ]:
# Cell 7 - Fit logistic regression on train, calibrate on cal
# Step 1: Fit base model on training data
base_model = LogisticRegression(random_state=SEED, max_iter=1000)
base_model.fit(X_train, y_train)

# Quick check on train (uncalibrated)
y_prob_train = base_model.predict_proba(X_train)[:, 1]
print(f'Train Brier (uncalibrated): {brier_score_loss(y_train, y_prob_train):.4f}')

# Step 2: Calibrate on 2022 calibration set
# sklearn 1.8: use FrozenEstimator to prevent re-fitting the base model
frozen = FrozenEstimator(base_model)
cal_model = CalibratedClassifierCV(frozen, method='isotonic')
cal_model.fit(X_cal, y_cal)

print(f'Calibration fitted on {len(X_cal)} observations (season 2022)')
print(f'Calibration method: isotonic regression')
print(f'\nBase model coefficients:')
for feat, coef in zip(FEATURES, base_model.coef_[0]):
    print(f'  {feat}: {coef:.4f}')
print(f'  intercept: {base_model.intercept_[0]:.4f}')

In [ ]:
# Cell 8 - Evaluate calibrated model on TEST set
y_prob_cal = cal_model.predict_proba(X_test)[:, 1]
y_pred_cal = (y_prob_cal >= 0.5).astype(int)

brier_cal = brier_score_loss(y_test, y_prob_cal)
logloss_cal = log_loss(y_test, y_prob_cal)
auc_cal = roc_auc_score(y_test, y_prob_cal)
f1_cal = f1_score(y_test, y_pred_cal, average='macro')

print('=== Calibrated Logistic Regression (Test Set 2023-2024) ===')
print(f'Brier Score:  {brier_cal:.4f}')
print(f'Log Loss:     {logloss_cal:.4f}')
print(f'ROC-AUC:      {auc_cal:.4f}')
print(f'Macro F1:     {f1_cal:.4f}')

print(f'\n=== Comparison vs Docent Baselines ===')
print(f'{"Model":<30} {"Brier":<10} {"ROC-AUC":<10}')
print('-' * 50)
print(f'{"Grid-rule (docent)":<30} {0.208:<10.4f} {"---":<10}')
print(f'{"Grid-rule (ours)":<30} {brier_grid:<10.4f} {auc_grid:<10.4f}')
print(f'{"Calibrated docent model":<30} {0.132:<10.4f} {0.892:<10.4f}')
print(f'{"Calibrated LogReg (ours)":<30} {brier_cal:<10.4f} {auc_cal:<10.4f}')

print(f'\n=== Honest Reflection ===')
if brier_cal < 0.132:
    print(f'Our model BEATS the calibrated docent baseline (Brier {brier_cal:.4f} < 0.132)')
elif brier_cal < 0.208:
    print(f'Our model beats grid-rule ({brier_cal:.4f} < 0.208) but not the calibrated docent (0.132).')
    print(f'Gap to docent: {brier_cal - 0.132:.4f}')
    print(f'\nThis is expected for a 3-feature linear model. The dominant signal is grid_position,')
    print(f'which alone explains most of the variance. The gap to the docent model likely comes from:')
    print(f'  1. Non-linear interactions (grid x constructor) that logistic regression cannot capture')
    print(f'  2. Additional features (driver form, circuit type) not yet included')
    print(f'  3. Possible use of gradient boosting in the docent model')
    print(f'\nHito 2 experiments are designed to close this gap with LightGBM and feature expansion.')
else:
    print(f'Our model does not beat grid-rule. Review needed.')

In [ ]:
# Cell 9 - Calibration curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration curve
ax = axes[0]
prob_true, prob_pred = calibration_curve(y_test, y_prob_cal, n_bins=8, strategy='uniform')
ax.plot(prob_pred, prob_true, 's-', label='Calibrated LogReg', color='#2196F3', linewidth=2)
prob_true_grid, prob_pred_grid = calibration_curve(y_test, y_prob_grid, n_bins=5, strategy='uniform')
ax.plot(prob_pred_grid, prob_true_grid, 'o--', label='Grid-rule', color='#FF9800', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability', fontsize=12)
ax.set_ylabel('Fraction of positives', fontsize=12)
ax.set_title('Calibration Curve (Test 2023-2024)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Histogram of predicted probabilities
ax = axes[1]
ax.hist(y_prob_cal[y_test == 1], bins=20, alpha=0.6, label='is_top10 = 1', color='#4CAF50')
ax.hist(y_prob_cal[y_test == 0], bins=20, alpha=0.6, label='is_top10 = 0', color='#F44336')
ax.set_xlabel('Predicted P(top10)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Predictions (Test 2023-2024)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Calibration curve saved.')

## What-If Scenario Comparison

In [ ]:
# Cell 10 - What-if scenarios
# Use UNCALIBRATED probabilities for what-if to show the raw model's sensitivity
# The calibrated model's isotonic mapping has limited bins, reducing sensitivity
scenarios = pd.DataFrame({
    'Scenario': [
        'A: LEC Monza P4, front, 1-stop M-H',
        'B: LEC Monza P4, front, 2-stop S-M-H',
        'C: Midfield P8, mid, 1-stop M-H',
        'D: Midfield P8, mid, 2-stop S-M-S',
        'E: Backmarker P15, back, 1-stop M-H',
        'F: Backmarker P15, back, 2-stop S-M-S'
    ],
    'grid_position': [4, 4, 8, 8, 15, 15],
    'constructor_tier': [2, 2, 1, 1, 0, 0],  # front=2, midfield=1, backmarker=0
    'n_stops': [1, 2, 1, 2, 1, 2]
})

X_scenarios = scenarios[FEATURES]
scenarios['P(top10)_calibrated'] = cal_model.predict_proba(X_scenarios)[:, 1]
scenarios['P(top10)_uncalibrated'] = base_model.predict_proba(X_scenarios)[:, 1]

print('=== What-If Scenario Comparison ===')
print(f'\n{"Scenario":<40} {"Grid":<6} {"Tier":<6} {"Stops":<6} {"P(cal)":<10} {"P(raw)":<10}')
print('-' * 78)
for _, row in scenarios.iterrows():
    tier_name = {2: 'front', 1: 'mid', 0: 'back'}[int(row['constructor_tier'])]
    print(f'{row["Scenario"]:<40} {int(row["grid_position"]):<6} {tier_name:<6} {int(row["n_stops"]):<6} {row["P(top10)_calibrated"]:<10.4f} {row["P(top10)_uncalibrated"]:<10.4f}')

print(f'\n--- Strategy Delta Analysis (uncalibrated probabilities) ---')
delta_AB = scenarios.iloc[1]['P(top10)_uncalibrated'] - scenarios.iloc[0]['P(top10)_uncalibrated']
delta_CD = scenarios.iloc[3]['P(top10)_uncalibrated'] - scenarios.iloc[2]['P(top10)_uncalibrated']
delta_EF = scenarios.iloc[5]['P(top10)_uncalibrated'] - scenarios.iloc[4]['P(top10)_uncalibrated']

print(f'Front-tier P4 (1-stop -> 2-stop): dP = {delta_AB:+.4f}')
print(f'Midfield P8 (1-stop -> 2-stop):   dP = {delta_CD:+.4f}')
print(f'Backmarker P15 (1-stop -> 2-stop): dP = {delta_EF:+.4f}')

print(f'\nNote: Small dP values reflect that n_stops has a small coefficient in the linear model.')
print(f'This is a known limitation - the model is dominated by grid_position.')
print(f'Hito 2 will explore non-linear models (LightGBM) that can capture strategy interactions.')

## Summary & Honest Reflection

In [ ]:
# Cell 11 - Final summary
print('=' * 60)
print('HITO 1 SUMMARY - GROUP 18')
print('=' * 60)
print(f'\nTarget: is_top10 (locked)')
print(f'Split:  Train 2019-2021 / Cal 2022 / Test 2023-2024')
print(f'Model:  Calibrated Logistic Regression (isotonic)')
print(f'Features: grid_position, constructor_tier, n_stops')
print(f'\n--- Test Set Results (2023-2024) ---')
print(f'Brier Score:  {brier_cal:.4f}')
print(f'Log Loss:     {logloss_cal:.4f}')
print(f'ROC-AUC:      {auc_cal:.4f}')
print(f'Macro F1:     {f1_cal:.4f}')
print(f'\n--- Docent Comparison ---')
print(f'Grid-rule floor (0.208):     {"BEATEN" if brier_cal < 0.208 else "NOT BEATEN"}')
print(f'Calibrated docent (0.132):   {"BEATEN" if brier_cal < 0.132 else "NOT BEATEN (gap: " + f"{brier_cal - 0.132:.4f}" + ")"}')
print(f'\n--- Honest Assessment ---')
print('Our 3-feature calibrated logistic regression provides a solid baseline')
print('that captures the dominant signal (grid position). The model is F1-defensible')
print('and was built without any reference to the test set.')
print('\nLimitations of this baseline:')
print('1. Linear model cannot capture grid x strategy interactions')
print('2. Only 3 features - no driver form or circuit characteristics')
print('3. n_stops has small marginal effect in a linear model')
print('4. Isotonic calibration with limited calibration data (426 obs) may lose granularity')
print('\nHito 2 plans: LightGBM, additional features, circuit-type stratification')